# Chess ML — Model Evaluation (Phase 5)

Evaluates the saved V2 model against a **deterministic held-out test set**.

The V2 training notebook used `ORDER BY RANDOM()` in SQL, so the exact training split
can't be reproduced. Instead we load positions ordered by `position_id` (stable),
apply the same 80/20 split, and evaluate. This gives reliable, reproducible numbers.

## Metrics
| Metric | Meaning |
|---|---|
| Top-1 accuracy | Correct move is the model's #1 pick |
| Top-3 accuracy | Correct move is in the model's top 3 |
| Top-5 accuracy | Correct move is in the model's top 5 |
| Legal-move coverage | % of positions where ≥1 legal move is known to encoder |
| Fallback rate | % of positions that need a random fallback move |

In [ ]:
import sys
import os
import pickle

import chess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sqlalchemy import text

# Allow importing db_connection from data/scripts
sys.path.insert(0, os.path.join('..', 'data', 'scripts'))
from db_connection import get_engine

print(f'TensorFlow: {tf.__version__}')
print(f'NumPy:      {np.__version__}')

## 1. Load model and label encoder

In [ ]:
MODEL_PATH   = os.path.join('..', 'models', 'chess_move_predictor_v2.keras')
ENCODER_PATH = os.path.join('..', 'models', 'label_encoder_v2.pkl')

model = tf.keras.models.load_model(MODEL_PATH)

with open(ENCODER_PATH, 'rb') as f:
    label_encoder = pickle.load(f)

print(f'Model loaded:   {MODEL_PATH}')
print(f'Encoder loaded: {ENCODER_PATH}')
print(f'Known moves:    {len(label_encoder.classes_):,}')
model.summary()

## 2. Load positions (deterministic order)

Using `ORDER BY position_id` so the 80/20 split is stable across runs.
Same ELO filter (>1400) as training.

In [ ]:
engine = get_engine()

print('Loading positions from database...')
df = pd.read_sql("""
    SELECT
        p.position_id,
        p.fen,
        p.move_played
    FROM positions p
    JOIN games g ON p.game_id = g.game_id
    WHERE g.white_elo > 1400
    ORDER BY p.position_id
""", engine)

print(f'Loaded {len(df):,} positions')
print(f'Unique moves: {df["move_played"].nunique():,}')
df.head(3)

## 3. Feature engineering

Same 781-feature encoding used during V2 training.

In [ ]:
PIECE_TYPES = [chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING]
COLORS = [chess.WHITE, chess.BLACK]

def fen_to_features(fen):
    board = chess.Board(fen)

    planes = np.zeros(768, dtype=np.float32)
    for color_idx, color in enumerate(COLORS):
        for piece_idx, piece_type in enumerate(PIECE_TYPES):
            plane = color_idx * 6 + piece_idx
            for sq in board.pieces(piece_type, color):
                planes[plane * 64 + sq] = 1.0

    side = np.array([1.0 if board.turn == chess.WHITE else 0.0], dtype=np.float32)

    castling = np.array([
        float(board.has_kingside_castling_rights(chess.WHITE)),
        float(board.has_queenside_castling_rights(chess.WHITE)),
        float(board.has_kingside_castling_rights(chess.BLACK)),
        float(board.has_queenside_castling_rights(chess.BLACK)),
    ], dtype=np.float32)

    ep = np.zeros(8, dtype=np.float32)
    if board.ep_square is not None:
        ep[chess.square_file(board.ep_square)] = 1.0

    return np.concatenate([planes, side, castling, ep])


print(f'Converting {len(df):,} positions to feature vectors...')
print('(~60-90 seconds)')

X = np.array([fen_to_features(fen) for fen in df['fen']])
y = label_encoder.transform(
    df['move_played'].where(df['move_played'].isin(label_encoder.classes_), other=label_encoder.classes_[0])
)
# Track which rows had an unknown move (not in encoder)
known_mask = df['move_played'].isin(label_encoder.classes_).values

print(f'Done. X shape: {X.shape}')
print(f'Moves known to encoder: {known_mask.sum():,} / {len(known_mask):,} ({known_mask.mean()*100:.1f}%)')

## 4. Deterministic 80/20 train/test split

In [ ]:
X_train, X_test, y_train, y_test, mask_train, mask_test = train_test_split(
    X, y, known_mask, test_size=0.2, random_state=42
)

print(f'Train: {len(X_train):,} samples')
print(f'Test:  {len(X_test):,} samples')

## 5. Compute Top-k accuracy on the test set

In [ ]:
print('Running inference on test set...')
y_pred_probs = model.predict(X_test, batch_size=512, verbose=1)

def topk_accuracy(y_true, y_pred_probs, k, mask=None):
    """Top-k accuracy, optionally restricted to rows where the true move is known."""
    if mask is not None:
        y_true       = y_true[mask]
        y_pred_probs = y_pred_probs[mask]
    top_k_preds = np.argsort(y_pred_probs, axis=1)[:, -k:]
    correct = np.any(top_k_preds == y_true[:, None], axis=1)
    return correct.mean()

# Evaluate only on positions where the true move is in the encoder vocabulary
top1 = topk_accuracy(y_test, y_pred_probs, 1, mask=mask_test)
top3 = topk_accuracy(y_test, y_pred_probs, 3, mask=mask_test)
top5 = topk_accuracy(y_test, y_pred_probs, 5, mask=mask_test)

v1_baseline = 0.0695

print()
print('=' * 55)
print('MODEL V2 EVALUATION RESULTS')
print('=' * 55)
print(f'  Test set size:     {mask_test.sum():,} positions (move in vocab)')
print(f'  Vocab coverage:    {mask_test.mean()*100:.1f}% of test positions')
print()
print(f'  Top-1 accuracy:    {top1*100:.2f}%   (V1: {v1_baseline*100:.2f}%)')
print(f'  Top-3 accuracy:    {top3*100:.2f}%')
print(f'  Top-5 accuracy:    {top5*100:.2f}%')
print()
print(f'  Improvement vs V1: {(top1-v1_baseline)*100:+.2f}pp  ({top1/v1_baseline:.1f}x)')
print('=' * 55)

## 6. Overfitting check — train vs test gap

In [ ]:
print('Running inference on training sample (5K) to check for overfitting...')
sample_idx = np.random.choice(len(X_train), size=5000, replace=False)
y_train_probs_sample = model.predict(X_train[sample_idx], batch_size=512, verbose=0)
mask_train_sample = mask_train[sample_idx]
y_train_sample = y_train[sample_idx]

train_top1 = topk_accuracy(y_train_sample, y_train_probs_sample, 1, mask=mask_train_sample)

gap = train_top1 - top1
print()
print('OVERFITTING CHECK')
print(f'  Train top-1:  {train_top1*100:.2f}%')
print(f'  Test  top-1:  {top1*100:.2f}%')
print(f'  Gap:          {gap*100:+.2f}pp  ', end='')
if gap < 0.03:
    print('✅ No significant overfitting')
elif gap < 0.08:
    print('⚠️  Mild overfitting')
else:
    print('❌ Significant overfitting — consider more dropout or less capacity')

## 7. Legal-move fallback rate

In production, the model masks illegal moves and falls back to a random legal move
if none of its known moves are legal. This cell measures how often that happens.

In [ ]:
SAMPLE_N = 500
sample_df = df.sample(n=SAMPLE_N, random_state=99)
known_classes = set(label_encoder.classes_)
fallback_count = 0

for fen in sample_df['fen']:
    try:
        board = chess.Board(fen)
        legal_sans = {board.san(m) for m in board.legal_moves}
        has_known = any(m in known_classes for m in legal_sans)
        if not has_known:
            fallback_count += 1
    except Exception:
        fallback_count += 1

fallback_rate = fallback_count / SAMPLE_N
print(f'Fallback rate: {fallback_count}/{SAMPLE_N} = {fallback_rate*100:.1f}%')
if fallback_rate < 0.05:
    print('✅ Very low — model handles nearly all positions')
elif fallback_rate < 0.15:
    print('⚠️  Moderate — may need more training data for rare positions')
else:
    print('❌ High — model vocabulary too narrow for production use')

## 8. Accuracy by move frequency

Does the model do better on common moves (e4, d4, Nf3) than rare ones?

In [ ]:
move_counts = df['move_played'].value_counts()

# Bin the test positions by how common their true move is
test_df_known = df.iloc[
    train_test_split(range(len(df)), test_size=0.2, random_state=42)[1]
].copy()
test_df_known = test_df_known[test_df_known['move_played'].isin(known_classes)].copy()
test_df_known['move_freq'] = test_df_known['move_played'].map(move_counts)

bins   = [0, 10, 50, 200, float('inf')]
labels = ['Rare (≤10)', 'Uncommon (11-50)', 'Common (51-200)', 'Very common (>200)']
test_df_known['freq_bin'] = pd.cut(test_df_known['move_freq'], bins=bins, labels=labels)

# Get predictions for these positions
X_known = np.array([fen_to_features(fen) for fen in test_df_known['fen']])
y_known_true = label_encoder.transform(test_df_known['move_played'])
y_known_pred = model.predict(X_known, batch_size=512, verbose=0)
top1_preds = np.argmax(y_known_pred, axis=1)
correct = (top1_preds == y_known_true)
test_df_known['correct'] = correct

acc_by_freq = test_df_known.groupby('freq_bin', observed=True)['correct'].agg(['mean', 'count'])
acc_by_freq.columns = ['Top-1 accuracy', 'Sample count']
acc_by_freq['Top-1 accuracy'] = acc_by_freq['Top-1 accuracy'].map('{:.1%}'.format)
print(acc_by_freq.to_string())

## 9. Visualise: accuracy by move frequency

In [ ]:
grp = test_df_known.groupby('freq_bin', observed=True)['correct'].mean()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(grp.index.astype(str), grp.values * 100, color='steelblue', edgecolor='white')
for bar, val in zip(bars, grp.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val*100:.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Top-1 Accuracy (%)')
ax.set_title('Model Accuracy by Move Frequency')
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Update stats endpoint with real numbers

Copy the values printed here into `app/routes.py` → `get_stats()`.

In [ ]:
print('Update app/routes.py get_stats() with these values:')
print()
print(f"    'top1_accuracy': {top1:.4f},")
print(f"    'top3_accuracy': {top3:.4f},")
print(f"    'top5_accuracy': {top5:.4f},")
print(f"    'fallback_rate': {fallback_rate:.4f},")